# Pandemic Signal Detective

When a new wave of disease starts, nobody announces it. You have to spot it in the data.

This notebook has real data from every US state, June 2020 to March 2022. Six different
kinds of data were tracking the same pandemic, but each one picks up a person at a
different stage of getting sick:

> infected → Google your symptoms → tell a survey you feel sick → see a doctor →
> test positive → go to hospital → some people die

So some of them should move earlier than others. Your job is to find out which one
warns us first, and then build an alarm with it.

| Signal | Where it comes from | What it counts |
|---|---|---|
| Searches | Google | Searches for loss of smell or taste |
| Survey | Facebook survey | Share of people who say they feel sick |
| Doctor visits | Insurance claims | Share of visits for COVID-like illness |
| Cases | Lab tests | Positive tests per 100,000 people |
| Hospital | Hospital reports | New COVID admissions per 100,000 |
| Deaths | Death records | COVID deaths per 100,000 |

Everything runs on dropdowns, checkboxes and sliders, so there is no code to write.
Run the cell below once, then work your way down.

In [ ]:
#@title Run this first { display-mode: "form" }
#@markdown Loads the data. Takes a few seconds. You only need to do this once.
DATA_URL = "https://signal-detective.github.io/covid_signals_by_state.csv" #@param {type:"string"}

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    try:
        pio.renderers.default = "colab"   # without this, charts come out blank
    except Exception:
        pass

DATA_FILE = "covid_signals_by_state.csv"
DELPHI_API = "https://api.delphi.cmu.edu/epidata/covidcast/"
START, END = "2020-06-01", "2022-03-31"
MIN_DAYS = 20

SIGNALS = {
    "searches": ("google-symptoms", "sum_anosmia_ageusia_smoothed_search"),
    "survey":   ("fb-survey", "smoothed_wcli"),
    "doctor":   ("doctor-visits", "smoothed_adj_cli"),
    "cases":    ("jhu-csse", "confirmed_7dav_incidence_prop"),
    "hospital": ("hhs", "confirmed_admissions_covid_1d_prop_7dav"),
    "deaths":   ("jhu-csse", "deaths_7dav_incidence_prop"),
}
LABELS = {"searches": "Searches", "survey": "Survey", "doctor": "Doctor visits",
          "cases": "Cases", "hospital": "Hospital", "deaths": "Deaths"}
COLORS = {"searches": "#7b5ea7", "survey": "#2a7fbf", "doctor": "#1f8a70",
          "cases": "#e08a1e", "hospital": "#c0392b", "deaths": "#44474f"}
EARLY_SIGNALS = ["searches", "survey", "doctor", "hospital", "deaths"]
PICK = {v: k for k, v in LABELS.items()}
WAVES = {
    "Winter 2020-21":          ("2020-10-01", "2021-02-28"),
    "Delta, summer 2021":      ("2021-06-15", "2021-10-31"),
    "Omicron, winter 2021-22": ("2021-11-15", "2022-03-15"),
    "Whole period":            ("2020-06-01", "2022-03-31"),
}
STATE_NAMES = {
    "AL":"Alabama","AK":"Alaska","AZ":"Arizona","AR":"Arkansas","CA":"California","CO":"Colorado",
    "CT":"Connecticut","DE":"Delaware","DC":"District of Columbia","FL":"Florida","GA":"Georgia",
    "HI":"Hawaii","ID":"Idaho","IL":"Illinois","IN":"Indiana","IA":"Iowa","KS":"Kansas",
    "KY":"Kentucky","LA":"Louisiana","ME":"Maine","MD":"Maryland","MA":"Massachusetts",
    "MI":"Michigan","MN":"Minnesota","MS":"Mississippi","MO":"Missouri","MT":"Montana",
    "NE":"Nebraska","NV":"Nevada","NH":"New Hampshire","NJ":"New Jersey","NM":"New Mexico",
    "NY":"New York","NC":"North Carolina","ND":"North Dakota","OH":"Ohio","OK":"Oklahoma",
    "OR":"Oregon","PA":"Pennsylvania","RI":"Rhode Island","SC":"South Carolina",
    "SD":"South Dakota","TN":"Tennessee","TX":"Texas","UT":"Utah","VT":"Vermont",
    "VA":"Virginia","WA":"Washington","WV":"West Virginia","WI":"Wisconsin","WY":"Wyoming",
}
STATE_CODE = {name: code for code, name in STATE_NAMES.items()}
NO_SEARCHES = "Alaska, DC, Delaware, Montana, North Dakota, South Dakota, Vermont and Wyoming"


def download_from_delphi(start=START, end=END):
    """One request per signal. Delphi allows 60 per hour per address, so six is safe."""
    import requests
    help_text = ("The Delphi API is refusing more requests from this address (it allows 60 "
                 "an hour, and a school shares one address). Ask your teacher for the data "
                 "file address, or upload covid_signals_by_state.csv using the folder icon.")
    frames = []
    for name, (src, sig) in SIGNALS.items():
        r = requests.get(DELPHI_API, timeout=180, params=dict(
            data_source=src, signals=sig, time_type="day", geo_type="state",
            geo_values="*", time_values=f"{start.replace('-', '')}-{end.replace('-', '')}"))
        if r.status_code == 429:
            raise RuntimeError(help_text)
        r.raise_for_status()
        try:
            payload = r.json()
        except ValueError:
            raise RuntimeError(f"Unreadable reply for '{name}'. " + help_text)
        rows = payload.get("epidata")
        if not rows:
            raise RuntimeError(f"No data came back for '{name}' "
                               f"({payload.get('message', 'no message')}). " + help_text)
        df = pd.DataFrame(rows)[["geo_value", "time_value", "value"]]
        df["signal"] = name
        frames.append(df)
    df = pd.concat(frames)
    df["date"] = pd.to_datetime(df.time_value.astype(str))
    wide = df.pivot_table(index=["geo_value", "date"], columns="signal",
                          values="value").reset_index()
    wide = wide.rename(columns={"geo_value": "state"})
    wide["state"] = wide.state.str.upper()
    wide[["state", "date"] + list(SIGNALS)].to_csv(DATA_FILE, index=False)


def load_signals():
    if os.path.exists(DATA_FILE):
        return pd.read_csv(DATA_FILE, parse_dates=["date"])
    if DATA_URL.strip():
        try:
            df = pd.read_csv(DATA_URL.strip(), parse_dates=["date"])
        except Exception as e:
            raise RuntimeError(
                f"Could not download the data file from:\n   {DATA_URL.strip()}\n"
                f"({type(e).__name__}: {e})\n\n"
                "Check the address with your teacher. On GitHub it has to be the 'raw' "
                "address. To use the Delphi API instead, clear the box and run again.") from None
        df.to_csv(DATA_FILE, index=False)
        return df
    download_from_delphi()
    return pd.read_csv(DATA_FILE, parse_dates=["date"])


data = load_signals()
_missing = [c for c in ["state", "date"] + list(SIGNALS) if c not in data.columns]
if _missing:
    raise RuntimeError(f"The data file is missing these columns: {_missing}")
data = data[data.state.isin(STATE_NAMES)]


# ---------- shared helpers ----------
def wave_slice(state, wave):
    start, end = WAVES[wave]
    return (data[(data.state == state) & (data.date >= start) & (data.date <= end)]
            .set_index("date").sort_index())


def usable(s, signal):
    return s[signal].notna().sum() >= MIN_DAYS


def rescale(series):
    lo, hi = series.min(), series.max()
    return (series - lo) / (hi - lo) * 100 if hi > lo else series * np.nan


def takeoff_date(series):
    """First day a signal climbs halfway to its peak, looking only before the peak."""
    s = rescale(series).dropna()
    if len(s) < MIN_DAYS:
        return None
    before = s.loc[:s.idxmax()]
    hits = before[before >= 50]
    return hits.index[0] if len(hits) else None


def best_lead(signal, cases, max_lag=21):
    """Days earlier (+) or later (-) a signal moves compared with cases."""
    best, best_lag = -np.inf, None
    for lag in range(-max_lag, max_lag + 1):
        r = signal.corr(cases.shift(-lag), min_periods=30)
        if pd.notna(r) and r > best:
            best, best_lag = r, lag
    return best_lag, best


def plural(n, word):
    return f"{n} {word}" if n == 1 else f"{n} {word}s"


def box(text, color="#2a7fbf", bg="#eef4fb"):
    display(HTML(f"<div style='background:{bg};border-left:4px solid {color};padding:9px 14px;"
                 f"margin:8px 0;font-size:15px'>{text}</div>"))


def gap_note(s, chosen, state, wave):
    gaps = [LABELS[k] for k in chosen if not usable(s, k)]
    if gaps:
        box(f"No data for <b>{', '.join(gaps)}</b> in {STATE_NAMES[state]} during {wave}, "
            f"so it is left out. Searches is missing for {NO_SEARCHES}, and stops in "
            f"January 2022.", "#d9a406", "#fdf6e3")


TABLE = [{"selector": "th", "props": [("background-color", "#2f3e49"), ("color", "white"),
                                      ("text-align", "left"), ("padding", "6px 12px")]},
         {"selector": "td", "props": [("padding", "6px 12px")]}]


# ---------- Activity 1 ----------
def race_to_the_top(state, wave, chosen):
    if not chosen:
        box("Tick at least one signal.", "#d9a406", "#fdf6e3")
        return
    s = wave_slice(state, wave)
    gap_note(s, chosen, state, wave)
    plotted = [k for k in chosen if usable(s, k)]
    if not plotted:
        box("Nothing you picked has data here. Try another place or wave.", "#d9a406", "#fdf6e3")
        return

    fig = go.Figure()
    rows = []
    case_takeoff = takeoff_date(s["cases"]) if usable(s, "cases") else None
    for sig in plotted:
        scaled = rescale(s[sig])
        fig.add_trace(go.Scatter(x=s.index, y=scaled, name=LABELS[sig],
                                 line=dict(color=COLORS[sig], width=3 if sig == "cases" else 2),
                                 hovertemplate="%{x|%b %d}: %{y:.0f}% of peak<extra></extra>"))
        t = takeoff_date(s[sig])
        if t is not None:
            fig.add_trace(go.Scatter(x=[t], y=[scaled.loc[t]], mode="markers", showlegend=False,
                                     marker=dict(symbol="star", size=15, color=COLORS[sig],
                                                 line=dict(color="white", width=1)),
                                     hovertemplate=f"{LABELS[sig]} takes off %{{x|%b %d}}<extra></extra>"))
            rows.append((LABELS[sig], t,
                         (case_takeoff - t).days if case_takeoff is not None else None))
    fig.update_layout(title=f"{STATE_NAMES[state]}, {wave}", height=430, template="plotly_white",
                      yaxis_title="% of its own peak", hovermode="x unified",
                      margin=dict(t=55, b=40), legend=dict(orientation="h", y=-0.17))
    fig.show()

    if not rows:
        box("Nothing reached halfway to a peak here, so there is no takeoff to rank.",
            "#d9a406", "#fdf6e3")
        return
    rows.sort(key=lambda r: r[1])
    first_date = rows[0][1]
    winners = [n for n, d, a in rows if d == first_date]
    who = (winners[0] if len(winners) == 1
           else ", ".join(winners[:-1]) + " and " + winners[-1])
    verb = "took off first" if len(winners) == 1 else "took off first together"
    ahead = rows[0][2]
    if ahead is not None and ahead > 0:
        box(f"<b>{who}</b> {verb} in {STATE_NAMES[state]}, "
            f"<b>{plural(ahead, 'day')}</b> before cases did.")
    else:
        box(f"<b>{who}</b> {verb}, on {first_date:%b %d}.")

    def vs_cases(name, a):
        if name == LABELS["cases"]:
            return "this is the comparison"
        if a is None:
            return ""
        return ("same day" if a == 0 else
                f"{plural(a, 'day')} earlier" if a > 0 else f"{plural(-a, 'day')} later")
    table = pd.DataFrame([(n, d.strftime("%b %d, %Y"), vs_cases(n, a)) for n, d, a in rows],
                         columns=["Signal", "Takes off", "Compared with cases"])
    table.index = [str(i + 1) for i in range(len(table))]
    display(table.style.set_table_styles(TABLE))


# ---------- Activity 2 ----------
def country_verdict(wave):
    rows = []
    for state in STATE_NAMES:
        s = wave_slice(state, wave)
        if not usable(s, "cases"):
            continue
        for sig in EARLY_SIGNALS:
            if not usable(s, sig):
                continue
            lag, r = best_lead(s[sig], s["cases"])
            if lag is not None and r > 0.5:
                rows.append(dict(state=STATE_NAMES[state], signal=LABELS[sig], key=sig, lead=lag))
    df = pd.DataFrame(rows)
    if df.empty:
        box("Nothing lined up closely enough with cases in this wave.", "#d9a406", "#fdf6e3")
        return
    med = df.groupby("signal").lead.median()
    order = med.sort_values(ascending=False).index.tolist()

    fig = px.strip(df, x="lead", y="signal", color="key", hover_name="state",
                   hover_data={"lead": True, "key": False, "signal": False},
                   color_discrete_map=COLORS, category_orders={"signal": order},
                   labels={"lead": "Days of warning before cases", "signal": ""})
    fig.add_trace(go.Scatter(x=med[order], y=order, mode="markers", name="median",
                             marker=dict(symbol="line-ns", size=26,
                                         line=dict(width=4, color="black"))))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(title=f"Every state, {wave}", height=400, template="plotly_white",
                      showlegend=False, margin=dict(t=55, b=45),
                      xaxis=dict(range=[-22, 22], dtick=7))
    fig.show()

    best, worst = order[0], order[-1]
    box(f"Each dot is one state. <b>{best}</b> warned earliest, by a median of "
        f"<b>{med[best]:+.0f} days</b>. <b>{worst}</b> came in at {med[worst]:+.0f} days, "
        f"so it moved <i>after</i> cases rather than before.")
    out = pd.DataFrame({"Median days of warning": med[order].round(0).astype(int),
                        "Places where it warned early":
                            df.groupby("signal").lead.apply(lambda x: f"{(x > 0).mean():.0%}")[order],
                        "Places measured": df.groupby("signal").lead.size()[order]})
    out.index.name = None
    display(out.style.set_table_styles(TABLE))


# ---------- Activity 3 ----------
def wave_map(signal, wave):
    start, end = WAVES[wave]
    df = data[(data.date >= start) & (data.date <= end)].copy()
    df["week"] = df.date.dt.to_period("W").dt.start_time
    weekly = df.groupby(["week", "state"])[signal].mean().reset_index().dropna()
    if weekly.empty:
        box(f"No {LABELS[signal]} data during {wave}.", "#d9a406", "#fdf6e3")
        return
    weekly["when"] = weekly.week.dt.strftime("%b %d, %Y")
    weekly["name"] = weekly.state.map(STATE_NAMES)
    if weekly.state.nunique() < len(STATE_NAMES):
        box(f"{LABELS[signal]} is only reported for {weekly.state.nunique()} of "
            f"{len(STATE_NAMES)} places. Grey means no data, not zero.", "#d9a406", "#fdf6e3")
    fig = px.choropleth(weekly, locations="state", locationmode="USA-states", scope="usa",
                        color=signal, animation_frame="when", hover_name="name",
                        color_continuous_scale="YlOrRd",
                        range_color=(weekly[signal].min(), weekly[signal].quantile(0.98)),
                        labels={signal: LABELS[signal]})
    fig.update_layout(title=f"{LABELS[signal]}, week by week", height=480,
                      margin=dict(t=55, b=10), coloraxis_colorbar=dict(title=""))
    if fig.layout.updatemenus and fig.layout.updatemenus[0].buttons:
        fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
    fig.show()


# ---------- Activity 4 ----------
def alarm_for(s, chosen, threshold, confirm_days=3):
    ratios = []
    for sig in chosen:
        x = s[sig]
        base = x.dropna().head(7).mean()
        if usable(s, sig) and pd.notna(base) and base > 0:
            ratios.append(x / base)
    if not ratios:
        return None, None
    combo = pd.concat(ratios, axis=1).mean(axis=1)
    above = (combo >= threshold).astype(int)
    streak = above.groupby((above != above.shift()).cumsum()).cumsum()
    hits = streak[streak >= confirm_days]
    return combo, (hits.index[0] if len(hits) else None)


def surge_date(s):
    c = rescale(s["cases"]).dropna()
    if c.empty:
        return None
    hits = c.loc[:c.idxmax()]
    hits = hits[hits >= 50]
    return hits.index[0] if len(hits) else None


def outbreak_alarm(state, wave, chosen, threshold, confirm_days):
    if not chosen:
        box("Tick at least one signal for your alarm to watch.", "#d9a406", "#fdf6e3")
        return
    s = wave_slice(state, wave)
    gap_note(s, chosen, state, wave)
    combo, alarm = alarm_for(s, chosen, threshold, confirm_days)
    if combo is None:
        box(f"Nothing you picked has data for {STATE_NAMES[state]} in this wave.",
            "#d9a406", "#fdf6e3")
        return
    surge = surge_date(s)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=combo.index, y=combo, name="your signal",
                             line=dict(color="#7b5ea7", width=3)))
    fig.add_trace(go.Scatter(x=s.index, y=s["cases"], name="cases", yaxis="y2",
                             line=dict(color=COLORS["cases"], width=2, dash="dot")))
    fig.add_hline(y=threshold, line_dash="dash", line_color="#7b5ea7",
                  annotation_text=f"{threshold:.1f}x")
    if alarm is not None:
        fig.add_vline(x=alarm, line_color="#c0392b", line_width=3)
        fig.add_annotation(x=alarm, y=1.06, yref="paper", text="alarm", showarrow=False,
                           font=dict(color="#c0392b", size=13))
    if surge is not None:
        fig.add_vline(x=surge, line_color=COLORS["cases"], line_width=3)
        fig.add_annotation(x=surge, y=0.96, yref="paper", text="surge", showarrow=False,
                           font=dict(color="#b36b0f", size=13))
    fig.update_layout(title=f"{STATE_NAMES[state]}, {wave}", height=420,
                      template="plotly_white", hovermode="x unified", margin=dict(t=55, b=40),
                      yaxis=dict(title="times its starting level"),
                      yaxis2=dict(title="cases per 100k", overlaying="y", side="right",
                                  showgrid=False),
                      legend=dict(orientation="h", y=-0.17))
    fig.show()

    if alarm is None:
        box("Your alarm never rang here. Lower the level, or ask for fewer days in a row.",
            "#d9a406", "#fdf6e3")
    elif surge is None:
        box("There was no clear surge in this state this wave, so there is nothing to beat.",
            "#d9a406", "#fdf6e3")
    else:
        d = (surge - alarm).days
        if d > 0:
            box(f"Your alarm rang <b>{d} days before</b> the surge in {STATE_NAMES[state]}.",
                "#1f8a70", "#eaf6f2")
        elif d < 0:
            box(f"Your alarm rang <b>{-d} days after</b> the surge had already started.",
                "#c0392b", "#fbeceb")
        else:
            box("Your alarm rang the same day the surge started.", "#d9a406", "#fdf6e3")

    leads, silent = [], 0
    for st in STATE_NAMES:
        ss = wave_slice(st, wave)
        sd = surge_date(ss)
        if sd is None:
            continue
        _, a = alarm_for(ss, chosen, threshold, confirm_days)
        if a is None:
            silent += 1
        else:
            leads.append((sd - a).days)
    leads = np.array(leads, dtype=float)
    early, total, twitchy = int((leads > 0).sum()), len(leads) + silent, int((leads > 45).sum())
    med = np.median(leads) if len(leads) else float("nan")
    if not len(leads):
        verdict = "It never rang anywhere. Lower the level."
    elif med < 0:
        verdict = "Too slow to be useful: on average it rings after the surge has started."
    elif twitchy > 6:
        verdict = f"Jumpy. It rings more than six weeks early in {twitchy} states, which in real life means crying wolf."
    elif med >= 10:
        verdict = "Solid. Early enough to act on, without going off constantly."
    else:
        verdict = "It works, but the warning is short. See if you can stretch it."
    display(pd.DataFrame({
        "": ["Rang before the surge", "Typical warning", "Never rang",
             "Rang more than 6 weeks early"],
        " ": [f"{early} of {total} places",
              f"{med:+.0f} days" if len(leads) else "n/a",
              plural(silent, "place"), plural(twitchy, "place")]
    }).style.hide(axis="index").set_table_styles(TABLE))
    box(verdict)


def picked(*pairs):
    return [k for k, on in pairs if on]


# ---------- quizzes ----------
UNANSWERED = "choose an answer"
QUIZ = {
    "star": ("the first day it climbed halfway to its peak",
             "That is the takeoff: the point where a signal is clearly on its way up.",
             "Look at where the stars sit on each line. Are they at the top, or partway up?"),
    "deaths_last": ("deaths reflect infections from weeks earlier",
                    "People who die were infected weeks before, so deaths report on a wave that already happened.",
                    "Think about how long it takes between catching COVID and dying of it."),
    "most_warning": ("Doctor visits",
                     "Doctor visits had the longest median lead in every wave, 5 to 12 days ahead of cases.",
                     "Read the chart top to bottom. The signal highest up has the biggest median."),
    "negative_lead": ("it moves about two weeks after cases",
                      "A negative lead means the signal follows cases instead of warning about them.",
                      "The dashed line is zero. What does it mean to sit to the left of it?"),
    "raise_level": ("it rings later, and in some places not at all",
                    "A stricter alarm waits for a bigger jump, so it fires late or misses the wave. Being sure and being early pull against each other.",
                    "Try it. Set the level to 3.0 and watch the 'never rang' row."),
    "claims_delay": ("insurance claims take days or weeks to reach anyone who could act",
                     "A signal is only as early as the moment it reaches you. Delphi had the dates, but in a real outbreak the paperwork lags.",
                     "This data is arranged by the date people visited. When does the insurance claim actually get processed?"),
}


def quiz(*pairs):
    out = []
    for key, given in pairs:
        answer, why, hint = QUIZ[key]
        if given == UNANSWERED:
            continue
        if given == answer:
            out.append(f"<div style='background:#eaf6f2;border-left:4px solid #1f8a70;"
                       f"padding:9px 14px;margin:8px 0'><b style='color:#1f8a70'>Correct.</b> {why}</div>")
        else:
            out.append(f"<div style='background:#fdf6e3;border-left:4px solid #d9a406;"
                       f"padding:9px 14px;margin:8px 0'><b style='color:#b3850a'>Not quite.</b> {hint}</div>")
    if out:
        display(HTML("".join(out)))
    else:
        display(HTML("<div style='color:#6b7280;font-size:14px;padding:4px'>"
                     "Pick an answer above and this will mark it.</div>"))


print(f"Ready. {len(data):,} rows, {data.state.nunique()} places (50 states and DC), "
      f"{data.date.min():%b %Y} to {data.date.max():%b %Y}.")

---
## 1. Which signal moves first?

Every line is scaled to a share of its own peak, so six different units fit on one chart.
The star marks each signal's takeoff: the first day it got halfway to its peak.

Start with your own state. Then check whether the same signal wins in a different wave.

In [ ]:
#@title Activity 1 { display-mode: "form", run: "auto" }
state = "California" #@param ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "District of Columbia", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
wave = "Winter 2020-21" #@param ["Winter 2020-21", "Delta, summer 2021", "Omicron, winter 2021-22", "Whole period"]
#@markdown Signals to show:
Searches = True #@param {type:"boolean"}
Survey = True #@param {type:"boolean"}
Doctor_visits = True #@param {type:"boolean"}
Cases = True #@param {type:"boolean"}
Hospital = True #@param {type:"boolean"}
Deaths = True #@param {type:"boolean"}

race_to_the_top(STATE_CODE[state], wave,
                picked(("searches", Searches), ("survey", Survey), ("doctor", Doctor_visits),
                       ("cases", Cases), ("hospital", Hospital), ("deaths", Deaths)))

In [ ]:
#@title Check your understanding { display-mode: "form", run: "auto" }
#@markdown **The star on each line marks...**
question_1 = "choose an answer" #@param ["choose an answer", "the day the signal peaked", "the first day it climbed halfway to its peak", "the day the wave ended"]
#@markdown **Deaths is almost always the last line to take off. Why?**
question_2 = "choose an answer" #@param ["choose an answer", "deaths reflect infections from weeks earlier", "death records are more accurate than the others", "fewer people died than caught COVID"]

quiz(("star", question_1), ("deaths_last", question_2))

---
## 2. Does it hold up everywhere?

One place could be luck. This runs the same comparison on all 50 states and DC at once, 51 in total.

For each state we slide a signal backwards and forwards in time until it best matches that
state's case curve. How far we had to slide it is its warning time. Dots to the right of
the dashed line warned early. Dots to the left moved after cases.

In [ ]:
#@title Activity 2 { display-mode: "form", run: "auto" }
wave = "Winter 2020-21" #@param ["Winter 2020-21", "Delta, summer 2021", "Omicron, winter 2021-22", "Whole period"]

country_verdict(wave)

In [ ]:
#@title Check your understanding { display-mode: "form", run: "auto" }
#@markdown **Which signal gave the most warning?**
question_1 = "choose an answer" #@param ["choose an answer", "Searches", "Survey", "Doctor visits", "Hospital"]
#@markdown **A signal with a warning time of -15 days is one where...**
question_2 = "choose an answer" #@param ["choose an answer", "it moves about two weeks after cases", "it warns us 15 days early", "the data is missing"]

quiz(("most_warning", question_1), ("negative_lead", question_2))

---
## 3. Watch a wave cross the country

Press play under the map. Try an early signal, then a late one, and watch which lights up
first.

In [ ]:
#@title Activity 3 { display-mode: "form", run: "auto" }
signal = "Survey" #@param ["Searches", "Survey", "Doctor visits", "Cases", "Hospital", "Deaths"]
wave = "Delta, summer 2021" #@param ["Winter 2020-21", "Delta, summer 2021", "Omicron, winter 2021-22"]

wave_map(PICK[signal], wave)

---
## 4. Build an alarm

Now make something useful out of it. Your alarm watches the signals you tick, and goes off
when they climb far enough above where they started and stay there.

Two dials. **Level** is how big a jump you demand. **Days in a row** is how long it has to
last before you believe it. Loose settings ring early but cry wolf. Strict settings are
trustworthy but slow. There is no setting that wins on both, which is the whole problem.

See how much warning you can get without the alarm going off six weeks early.

In [ ]:
#@title Activity 4 { display-mode: "form", run: "auto" }
state = "Texas" #@param ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "District of Columbia", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
wave = "Delta, summer 2021" #@param ["Winter 2020-21", "Delta, summer 2021", "Omicron, winter 2021-22"]
#@markdown Signals your alarm watches:
Searches = True #@param {type:"boolean"}
Survey = True #@param {type:"boolean"}
Doctor_visits = True #@param {type:"boolean"}
#@markdown How big a jump before it counts:
level = 1.5 #@param {type:"slider", min:1.1, max:4.0, step:0.1}
#@markdown How many days in a row it must stay up:
days_in_a_row = 3 #@param {type:"slider", min:1, max:10, step:1}

outbreak_alarm(STATE_CODE[state], wave,
               picked(("searches", Searches), ("survey", Survey), ("doctor", Doctor_visits)),
               level, days_in_a_row)

In [ ]:
#@title Check your understanding { display-mode: "form", run: "auto" }
#@markdown **You raise the level from 1.5 to 3.0. What happens to your alarm?**
question_1 = "choose an answer" #@param ["choose an answer", "it rings later, and in some places not at all", "it rings earlier in every place", "nothing changes"]
#@markdown **Doctor visits won on this data. Why might you still not trust it in a real outbreak?**
question_2 = "choose an answer" #@param ["choose an answer", "insurance claims take days or weeks to reach anyone who could act", "doctors are bad at spotting COVID", "there is not enough data"]

quiz(("raise_level", question_1), ("claims_delay", question_2))

---
## Worth discussing

**Searches has no data at all for eight of the 51 places,** and the survey only reached people on
Facebook. Cases only counted people who got a test. Whose illness does each one miss?

**In 2013 Google Flu Trends badly overshot,** partly because news coverage sent healthy
people searching for symptoms. Which of our six signals could break the same way, and when?

**This data came from millions of people's searches, surveys and medical records,**
anonymised and pooled. What would you want to be true before your own data was used this way?

---
<sub>Data from the [Delphi Epidata API](https://cmu-delphi.github.io/delphi-epidata/) at
Carnegie Mellon, which gathered these signals from Google, Facebook, health systems, Johns
Hopkins and the US Department of Health and Human Services. The assembled file lives at
[github.com/signal-detective/signal-detective.github.io](https://github.com/signal-detective/signal-detective.github.io).</sub>

<sub>**Teacher:** every panel is a Colab form with the code folded away. Double-click a
title bar to show the Python behind it. The data loads from the GitHub copy above in one
download per student, which matters because Delphi only allows 60 requests an hour per
address and a school shares one.</sub>